# Auswertung Zelasto Studie

![Bild Depth Touch](images/depth-touch.jpg)

Jupyter Notebook um die in der Zelasto Studie geschriebenen CSV Dateien auf und auszuwerten. 

1. [Studienablauf](#Studienablauf)
1. [Aktuelle Lösung](Aktuelle-Lösung)
1. [Struktur Logfiles](#Struktur-Logfiles)
1. [Werte im Logfile](#Werte-im-Logfile)
1. [Statusabfolge](#Statusabfolge)
1. [Definition der Tiefenebenen](#Definition-der-Tiefenebenen)

## Studienablauf
---

* 6 Probe-Aufgaben (TASK) mit jeweils 2 Wiederholungen (TRIAL)
* 3 Blöcke (BLOCK) mit jeweils
    * 18 Aufgaben (TASK) mit jeweils 5 Wiederholungen (TRIAL)
    * In dem Block wurde immer eine Mapping-Methode (MAPPING_METHOD) genutzt
        * `direct / densening / widening`
    * Variiert wurden in den Aufgaben (TASK) die Anzahl der Tiefenebenen:
        * `6 / 9 / 12 / 15 / 18 / 21`
    * Variiert in den Wiederholungen (TRIALS) wurden die Zielebenen (random)
* Zuordnung Ebene <-> MAPPING_METHOD <-> Tiefe in eigener csv (depthlayers.csv)
* Erfolgs-/Fehlermöglichkeiten:
    * `COMPLETED / FAILED / TERMINATED`

## Aktuelle Lösung
---

* Iterativer / objektorientierter Ansatz (nicht wirklich Data Science strukturiert)
* Gesucht: besser anpassbare, „einfache“ data science Lösung

## Struktur Logfiles
---

`2021-05-11T14:55:08.419Z; INTERACTION; direct;18;14,9,5,1,17;18;1; 0.5099127;0.5482645;-0.449999869;637563489084131200;1;8`

| DateTime (LogServer)     | STATE          | mapping method | Task-No. | Target Layers | Layer-Count | Trial-Idx | PosX             | PosY                                                  | PosZ         | TimeStamp (Server) | InteractionType | CurrentLayer |
| ------------------------ | -------------- | -------------- | -------- | ------------- | ----------- | --------- | ---------------- | ----------------------------------------------------- | ------------ | ------------------ | --------------- | ------------ |
| 2021-05-11T14:55:08.419Z | INTERACTION    | direct         | 18       | 14,9,5,1,17   | 18          | 1         | 0.5099127        | 0.5482645                                             | -0.449999869 | 637563489084131200 | 1               | 8            |
| 2021-05-10T12:14:37.259Z | VIEW           | direct         | 2        | 7,8,1,5,2     | 9           | 0         | TASK_DESCRIPTION |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.259Z | TASK           | direct         | 2        | 7,8,1,5,2     | 9           | 0         |                  |                                                       |              |                    |                 |              |
| 2021-05-10T12:13:51.730Z | MAPPING_METHOD | direct         | 1        | 4,5,3,1,2     | 6           | 0         |                  |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.264Z | SUBTASK        | direct         | 2        | 7,8,1,5,2     | 9           | 0         | 2                |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:37.265Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | START            |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:39.587Z | VIEW           | direct         | 2        | 7,8,1,5,2     | 9           | 0         | TASK_VIEW        |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:45.064Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | HOLD             |                                                       |              |                    |                 |              |
| 2021-05-10T12:14:46.565Z | SUBTASK_STATE  | direct         | 2        | 7,8,1,5,2     | 9           | 0         | COMPLETED        |                                                       |              |                    |                 |              |
| 2021-05-10T12:10:39.115Z | SUBTASK_STATE  | direct         | 1        | 1,5           | 6           | 0         | FAILED           | wrong level approved                                  |              |                    |                 |              |
| 2021-05-10T12:13:38.243Z | SUBTASK_STATE  | densening      | 6        | 3,12          | 14          | 0         | TERMINATED       | switched to other level before hold timeout completed |              |                    |                 |              |

## Werte im Logfile
---

* Tasks:
    * 1-6 (Test)
    * 1-18 (Block 1)
    * 19-36 (Block 2)
    * 37 - 54 (Block 3)

* 3 Blocks for any mapping method

* STATE:
  
    | State Value        | Description                                       | SubTypes           |
    | ------------------ | ------------------------------------------------- | ------------------ |
    | __INTERACTION__    | Trial interaction                                 | -                  |
    | __VIEW__           | switched view (describe Task)                     | TASK_VIEW          |
    |                    | switched view in test runs (describe Task)        | Test Run TASK_VIEW |
    |                    |                                                   | TASK_DESCRIPTION   |
    | __TASK__           |                                                   |                    |
    | __MAPPING_METHOD__ | starting to next large block (mapping method)     |                    |
    | __SUBTASK__        | same as the next: starting to next task           |                    |
    | __SUBTASK_STATE__  | start of the trial after Subtask                  | START              |
    |                    | dwell time in a layer exceeded: hold-timer starts | HOLD               |
    |                    | end of the trial (Success)                        | COMPLETED          |
    |                    | end of the trial (hold failure                    | TERMINATED         |
    |                    | end of the trial (wrong level ?)                  | FAILED             |

* mapping method - describes, how layers are aligned:
    * __direct__ (equivalent distance)
    * __densening__ (larger distance on top, decreasing with depth value)
    * __widening__ (narrower distance on top, increasing with depth value)

* Task-No.
    * running number of tasks

* Target Layers, Trial-Idx
    * array of targets for each layer in current Task
    * trial-Index (zero based) describes target layer for current trial

* Layer-Count
    * number of max layers in Task

* PosX, PosY, PosZ
    * Positions received from Tracking Server
    * PosX / PosY in range [0, 1]
    * PosZ in range [-1, 1] with 0 = on the surface / no interaction, -1 max push, +1 max pull

* Timestamp (Server)
    * timestamp received from Tracking server: miliseconds based unix time stamp

* InteractionType
    * type recognized from server (1 = PUSH)

* current layer
    * layer associated with received depth value

## Statusabfolge
---

1. START (missing on first trial !) OR
    * start of an new task
2. TASK_VIEW / TASK_DESCRIPTION
3. HOLD
4. COMPLETED / TERMINATED / FAILED
   

## Definition der Tiefenebenen
---

__Location:__ `data/depthlayers.csv` [File](data/depthlayers.csv)

`6; direct; 0 | 1 | 2 | 3 | 4 | 5 | 6; 0 | 0.0834 | 0.25 | 0.4167 | 0.5834 | 0.75 | 0.9167 | 1`

| NUM_LAYERS       | MAPPING_METHOD            | DEPTH_LAYER_ID                         | DEPTH_LAYER_BORDER              |
| ---------------- | ------------------------- | -------------------------------------- | ------------------------------- |
| number of layers | mapping method for  block | idx of the depth layer in border array | start depths for each layer idx |

* mapping of layers to depth ranges for better evaluation how "good" a depth layer has been hit


## Aufgaben
---

### Vorverarbeitung - Cleaning

* Probe-Aufgaben und eigentliche Studie in verschiedene Dateien
* Ersetzen: Pro Task, muss im ersten Trial  TASK_VIEW mit START getauscht werden
* Spalten benennen (für bessere Adressierung)
* Nebenbedingung/Bonus: sowohl „nur Studie“ / „nur Test“ / „Test und Studie“ zusammen laden  Ordner-Struktur / Namensschema ?
* Lösung/Code dokumentieren (jupyter notebook)


In [1]:
# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob

path = "../data/"
all_files = glob.glob(path + "/*.csv")

# for debugging: only use first file
# all_files = all_files[9:11]

study =[]
cleanedStudy = []
columnNames =["DateTime", "State", "mappingMethod", "TaskNo", "TargetLayers", "LayerCount", "TrialIdx", "posX", "posY", "posZ", "TimeStamp", "iteractionType", "currentLayer"]
#testData = pd.read_csv("../data/01.csv", sep=";", names = columnNames) #load csv data with delimeter and add column names ;

totalLength = 0

for filename in all_files:
    df =pd.read_csv(filename, sep=";", names = columnNames)
    study.append(df)
    totalLength += len(df)
    
print("totallines: " + str(totalLength))

currentLine = 0
idx = 0

marker = []
marker.append(["Id", "EndInit", "EndTests", "SwapLines"])

for testData in study:
    endInit=0
    endTest=0
    
    idx += 1
    currentLine += len(testData)

    # drop data from the start and end of the file, that are not needed
    for i in range(1, len(testData)-2):                                        #iterate through the file to find the lines were the actual test starts ans ends
        if (testData["TaskNo"][i] == 6) and (testData["TaskNo"][i+1] == 1):
            endInit = i+1
        if (testData["TaskNo"][i] == 54) and (testData["TaskNo"][i+1] == 1):
            endTests = i+1

    marker.append([idx, endInit, endTests, "-"]);
    
    print("Processed: " + str(currentLine) + " of " +  str(totalLength) + "(" + str(idx) + ")")
    
    filteredData = testData.drop(testData.index[endTests:len(testData)])     
    filteredData = filteredData.drop(filteredData.index[0:endInit])

    filteredData.reset_index(drop=True, inplace=True)

    # exchange TASK_VIEW ans START in the first Trial for each Task
    startLine =0
    taskLine=0

    # iterate over all Lines to find START and TASK_VIEW 
    # Swap lines when TASK_VIEW was found
    for i in range(0, len(filteredData)-2):
        if filteredData["TrialIdx"][i] == 0 and filteredData["posX"][i] == " START":
            startLine = i
        if filteredData["TrialIdx"][i] == 0 and filteredData["posX"][i] == " TASK_VIEW":
            taskLine = i
            linecopy = filteredData.iloc[startLine]
            filteredData.iloc[startLine] = filteredData.iloc[taskLine]
            filteredData.iloc[taskLine] = linecopy
            
            marker.append([idx, "-", "-", str(startLine + endInit) + " <-> " + str(taskLine + endInit)]);

    cleanedStudy.append(filteredData)
    
markerframe = pd.DataFrame(data=marker)
markerframe.to_csv(r'marker.csv', sep= ";")
    
print('finished')

totallines: 2733080
Processed: 86106 of 2733080(1)
Processed: 227176 of 2733080(2)
Processed: 339055 of 2733080(3)
Processed: 478493 of 2733080(4)
Processed: 594798 of 2733080(5)
Processed: 720103 of 2733080(6)
Processed: 883086 of 2733080(7)
Processed: 1064088 of 2733080(8)
Processed: 1184333 of 2733080(9)
Processed: 1299948 of 2733080(10)
Processed: 1391165 of 2733080(11)
Processed: 1490464 of 2733080(12)
Processed: 1674368 of 2733080(13)
Processed: 1777237 of 2733080(14)
Processed: 1904586 of 2733080(15)
Processed: 2014642 of 2733080(16)
Processed: 2120240 of 2733080(17)
Processed: 2227439 of 2733080(18)
Processed: 2343047 of 2733080(19)
Processed: 2444664 of 2733080(20)
Processed: 2543798 of 2733080(21)
Processed: 2634877 of 2733080(22)
Processed: 2733080 of 2733080(23)
finished


---

### Extraktion

#### Erfolg/Fehler für jeden Trial und je Proband

| PROBAND | BLOCK | TASK | TRIAL | MAPPING_METHOD | NUM_LAYERS | TARGET | TARGET_RELATIVE | RESULT    |
| ------- | ----- | ---- | ----- | -------------- | ---------- | ------ | --------------- | --------- |
| 1       | 1     | 2    | 5     | direct         | 15         | 7      | 0.46666         | COMPLETED |
| .       | .     | .    | .     | .              | .          | .      | .               | .         |
| .       | .     | .    | .     | .              | .          | .      | .               | .         |

---

In [2]:
import math
#create new Dataframe for the results per Trial and Proband
cn_result = ["Proband", "Block", "Task", "Trial", "MappingMethod", "NumLayers", "Target", "Target_Relative", "Result", "Result_Numeric"]
endOfTrials = pd.DataFrame(columns = cn_result) 
probandNum = 0

# iterate through all probands
for proband in cleanedStudy:
    last_start = len(proband)
    
    # search for all results for each proband   
    for i in range (0, len(proband)):
        
        # to prevent "double results": note index of last_start
        if proband["posX"][i] == " START":
            last_start = i
        
        if (proband["posX"][i] == " COMPLETED" or proband["posX"][i] == " FAILED" or proband["posX"][i] == " TERMINATED"):
            # add Result to Dataframe
            # copy get the needed values and addd entry to the dataframe
            blockNo = math.ceil(proband["TaskNo"][i] / 18)
            taskNo = proband["TaskNo"][i] - ((blockNo - 1) * 18)
            layerCount = proband["LayerCount"][i]
            target = int(proband["TargetLayers"][i].split(",")[proband["TrialIdx"][i]])
            numeric_result = 0
            if proband["posX"][i] == " COMPLETED":
                numeric_result = 1
            
            if last_start >= i:
                # double result - message
                print("double result found for proband " + str(probandNum) + " : [" + str(blockNo) + " | " + str(taskNo) + " | " + str(proband["TrialIdx"][i]) + " | " + str(i) + "]")
            else: 
                endOfTrials.loc[len(endOfTrials)] = [probandNum, blockNo, taskNo, proband["TrialIdx"][i], proband["mappingMethod"][i], layerCount, target, target / layerCount, proband["posX"][i], numeric_result ]
            
                last_start = len(proband)
            
    probandNum = probandNum+1
    print("Processed: " + str(probandNum) + " of " + str(len(cleanedStudy)))
display(endOfTrials)

endOfTrials.to_csv(r'results.csv', sep= ";")



Processed: 1 of 23
Processed: 2 of 23
Processed: 3 of 23
Processed: 4 of 23
Processed: 5 of 23
Processed: 6 of 23
Processed: 7 of 23
Processed: 8 of 23
Processed: 9 of 23
double result found for proband 9 : [3 | 8 | 2 | 71983]
Processed: 10 of 23
Processed: 11 of 23
Processed: 12 of 23
Processed: 13 of 23
Processed: 14 of 23
Processed: 15 of 23
Processed: 16 of 23
Processed: 17 of 23
Processed: 18 of 23
Processed: 19 of 23
Processed: 20 of 23
Processed: 21 of 23
Processed: 22 of 23
Processed: 23 of 23


,Proband,Block,Task,Trial,MappingMethod,NumLayers,Target,Target_Relative,Result,Result_Numeric
0,0,1,1,0,direct,6,4,0.666667,COMPLETED,1
1,0,1,1,1,direct,6,5,0.833333,TERMINATED,0
2,0,1,1,2,direct,6,3,0.500000,COMPLETED,1
3,0,1,1,3,direct,6,1,0.166667,TERMINATED,0
4,0,1,1,4,direct,6,2,0.333333,COMPLETED,1
...,...,...,...,...,...,...,...,...,...,...
6205,22,3,18,0,widening,6,4,0.666667,COMPLETED,1
6206,22,3,18,1,widening,6,5,0.833333,COMPLETED,1
6207,22,3,18,2,widening,6,2,0.333333,TERMINATED,0
6208,22,3,18,3,widening,6,3,0.500000,COMPLETED,1


---
#### Versuchsdauer für jeden Trial und je Proband

| PROBAND | BLOCK | TASK | TRIAL | MAPPING_METHOD | NUM_LAYERS | TARGET | DURATION |
| ------- | ----- | ---- | ----- | -------------- | ---------- | ------ | ------   |
| 1       | 1     | 2    | 5     | direct         | 15         | 7      | 15.532s  |
| .       | .     | .    | .     | .              | .          | .      | .        |
| .       | .     | .    | .     | .              | .          | .      | .        |

---

In [6]:
from datetime import datetime, timedelta

#create new Dataframe for the results per Trial and Proband
cn_times = ["Proband", "Block", "Task", "Trial", "MappingMethod", "NumLayers", "Target", "Target_Relative", "Duration"]
end_times = pd.DataFrame(columns = cn_times) 
probandNum = 0

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

# iterate through all probands
for proband in cleanedStudy:
    last_start = len(proband)
    time_start = datetime(1,1,1)

    # search for all results for each proband   
    for i in range (0, len(proband)):       
        
        # to prevent "double results": note index of last_start
        # save starting time
        if proband["posX"][i] == " START":
            last_start = i
            time_start = datetime.strptime(proband["DateTime"][i], timeFormat)
            
        if proband["posX"][i] == " COMPLETED" or proband["posX"][i] == " FAILED" or proband["posX"][i] == " TERMINATED":
            # add Result to Dataframe
            # copy get the needed values and addd entry to the dataframe
            blockNo = math.ceil(proband["TaskNo"][i] / 18)
            taskNo = proband["TaskNo"][i] - ((blockNo - 1) * 18)
            layerCount = proband["LayerCount"][i]
            target = int(proband["TargetLayers"][i].split(",")[proband["TrialIdx"][i]])
            time_end = datetime.strptime(proband["DateTime"][i], timeFormat)

            delta = (time_end - time_start).total_seconds()
            
            if last_start >= i:
                # double result - message
                print("double result found for proband " + str(probandNum) + " : [" + str(blockNo) + " | " + str(taskNo) + " | " + str(proband["TrialIdx"][i]) + " | " + str(i) + "]")
            else: 
            
                end_times.loc[len(end_times)] = [probandNum, blockNo, taskNo, proband["TrialIdx"][i], proband["mappingMethod"][i], layerCount, target, target / layerCount, delta ]
            
                time_start = datetime(1,1,1)
                time_end = datetime(1,1,1)            
                last_start = len(proband)

    probandNum = probandNum+1
    print("Processed: " + str(probandNum) + " of " + str(len(cleanedStudy)))

display(end_times)

end_times.to_csv(r'times.csv', sep= ";")

Processed: 1 of 23
Processed: 2 of 23
Processed: 3 of 23
Processed: 4 of 23
Processed: 5 of 23
Processed: 6 of 23
Processed: 7 of 23
Processed: 8 of 23
Processed: 9 of 23
double result found for proband 9 : [3 | 8 | 2 | 71983]
Processed: 10 of 23
Processed: 11 of 23
Processed: 12 of 23
Processed: 13 of 23
Processed: 14 of 23
Processed: 15 of 23
Processed: 16 of 23
Processed: 17 of 23
Processed: 18 of 23
Processed: 19 of 23
Processed: 20 of 23
Processed: 21 of 23
Processed: 22 of 23
Processed: 23 of 23


,Proband,Block,Task,Trial,MappingMethod,NumLayers,Target,Target_Relative,Duration
0,0,1,1,0,direct,6,4,0.666667,12.221
1,0,1,1,1,direct,6,5,0.833333,4.414
2,0,1,1,2,direct,6,3,0.500000,7.155
3,0,1,1,3,direct,6,1,0.166667,3.627
4,0,1,1,4,direct,6,2,0.333333,5.516
...,...,...,...,...,...,...,...,...,...
6205,22,3,18,0,widening,6,4,0.666667,7.359
6206,22,3,18,1,widening,6,5,0.833333,5.325
6207,22,3,18,2,widening,6,2,0.333333,3.438
6208,22,3,18,3,widening,6,3,0.500000,5.467


#### Versuchsdauer für jeden Trial und je Proband (Trial als eigene Spalten)

| PROBAND | BLOCK | TASK | TRIAL0 | ... | TRIALN | MAPPING_METHOD | NUM_LAYERS | TARGET | DURATION |
| ------- | ----- | ---- | ------ | --- | -------| -------------- | ---------- | ------ | ------   |
| 1       | 1     | 2    | 5.532s | ... | 2.546s | direct         | 15         | 7      | 15.532s  |
| .       | .     | .    | .      | ... | .      | .              | .          | .      | .        |
| .       | .     | .    | .      | ... | .      | .              | .          | .      | .        |

In [123]:
pivot = end_times.pivot(index=["Proband", "Block", "Task", "MappingMethod", "NumLayers"], columns="Trial", values="Duration")

# print(end_times.columns)
# print(end_times.index)

# display(pivot.columns.names)
# print(pivot.index.names)

# creating the multiIndex... 
m_idx = pd.MultiIndex.from_frame(pivot)

print(m_idx.get_level_values(0))
    
pivot["Duration_Sum"] = m_idx.get_level_values(0) +  m_idx.get_level_values(1) +  m_idx.get_level_values(2) +  m_idx.get_level_values(3) +  m_idx.get_level_values(4)

display(pivot)

pivot.to_csv(r'times_repeat.csv',sep= ";")

Float64Index([12.221,    9.3, 11.357, 10.526,  9.006,  9.445, 12.669,   8.59,
               8.982, 10.104,
              ...
               7.538,  9.748,  7.919,  9.042, 12.942,  7.163,  10.43, 10.177,
              12.128,  7.359],
             dtype='float64', name=0, length=1242)


Trial                                            0       1       2       3  \
Proband Block Task MappingMethod NumLayers                                   
0       1     1     direct       6          12.221   4.414   7.155   3.627   
              2     direct       9           9.300   6.029  13.550  11.628   
              3     direct       9          11.357   7.849   6.402   4.394   
              4     direct       15         10.526   7.910   7.202   7.145   
              5     direct       12          9.006   8.612   8.256   5.049   
...                                            ...     ...     ...     ...   
22      3     14    widening     9           7.163   4.929   5.777   6.530   
              15    widening     12         10.430   5.419   8.240   5.712   
              16    widening     12         10.177  15.856   6.571   6.755   
              17    widening     21         12.128   9.413   4.488   4.400   
              18    widening     6           7.359   5.325   3.438   5.467   

Trial                                           4  Duration_Sum  
Proband Block Task MappingMethod NumLayers                       
0       1     1     direct       6          5.516        32.933  
              2     direct       9          9.832        50.339  
              3     direct       9          8.062        38.064  
              4     direct       15         7.956        40.739  
              5     direct       12         4.624        35.547  
...                                           ...           ...  
22      3     14    widening     9          5.683        30.082  
              15    widening     12         6.656        36.457  
              16    widening     12         5.323        44.682  
              17    widening     21         6.644        37.073  
              18    widening     6          5.088        26.677  

[1242 rows x 6 columns]

---
#### Erfolg/Fehlerquote über alle Probanden je Ebenenanzahl

| NUM_LAYERS     | COMPLETED | FAILED  | TERMINATED | SUM |
| ---            | ---       | ---     | ---        | --- |
| 5              | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .              | .         | .       | .          | .   |
| .              | .         | .       | .          | .   |
---

In [301]:
df_resultStat = pd.read_csv('results.csv', sep=";", names = cn_result)
df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["NumLayers", "Result"]) 

groups = df_resultStat.groupby(["NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0], 0, 0, 0, 0, 0, 0, 0]

count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["NumLayers"] == index[0])]
    r_index = rate.index
        
    if index[1] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 1] = compl_sum
        
    if index[1] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 3] = fail_sum
        
    if index[1] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 5] = term_sum
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by="NumLayers", key=lambda col: col.astype(int)))

df_rates.to_csv(r'rates_global_layers.csv', sep= ";")

,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
4,6,739,0.71401,13,0.01256,283,0.27343,1035,1.0
5,9,774,0.747826,16,0.015459,245,0.236715,1035,1.0
0,12,732,0.707246,16,0.015459,287,0.277295,1035,1.0
1,15,717,0.692754,26,0.025121,292,0.282126,1035,1.0
2,18,708,0.684058,26,0.025121,301,0.290821,1035,1.0
3,21,679,0.656039,38,0.036715,318,0.307246,1035,1.0


---
#### Erfolg/Fehlerquote über je Probanden je Ebenenanzahl

| Proband     | NUM_LAYERS     | COMPLETED | FAILED  | TERMINATED | SUM |
| ---         | ---            | ---       | ---     | ---        | --- |
| 0           | 5              | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .           | .              | .         | .       | .          | .   |
| .           | .              | .         | .       | .          | .   |
---

In [302]:
df_resultStat = pd.read_csv('results.csv', sep=";", names = cn_result)
df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["Proband", "NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["Proband", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["Proband", "NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], 0, 0, 0, 0, 0, 0, 0]

    
count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["Proband"] == index[0]) & (df_rates["NumLayers"] == index[1])]
    r_index = rate.index
        
    if index[2] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 2] = compl_sum
        
    if index[2] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 4] = fail_sum
        
    if index[2] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 6] = term_sum
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by=["Proband", "NumLayers"], key=lambda col: col.astype(int)))

df_rates.to_csv(r'rates_subject_layers.csv', sep= ";")

,Proband,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
4,0,6,27,0.6,0,0.0,18,0.4,45,1.0
5,0,9,31,0.688889,0,0.0,14,0.311111,45,1.0
0,0,12,29,0.644444,1,0.022222,15,0.333333,45,1.0
1,0,15,33,0.733333,2,0.044444,10,0.222222,45,1.0
2,0,18,31,0.688889,1,0.022222,13,0.288889,45,1.0
...,...,...,...,...,...,...,...,...,...,...
95,22,9,31,0.688889,4,0.088889,10,0.222222,45,1.0
90,22,12,30,0.666667,2,0.044444,13,0.288889,45,1.0
91,22,15,27,0.6,4,0.088889,14,0.311111,45,1.0
92,22,18,27,0.6,7,0.155556,11,0.244444,45,1.0


---
#### Erfolg/Fehlerquote über alle Probanden je Ebenenanzahl und mapping methode

| MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED  | TERMINATED | SUM |
| ---            | ---        | ---       | ---     | ---        | --- |
| direct         | 5          | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .              | .          | .         | .       | .          | .   |
| .              | .          | .         | .       | .          | .   |
---

In [303]:
df_resultStat = pd.read_csv('results.csv', sep=";", names = cn_result)
df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["MappingMethod", "NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["MappingMethod", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["MappingMethod", "NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], 0, 0, 0, 0, 0, 0, 0]

    
count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["MappingMethod"] == index[0]) & (df_rates["NumLayers"] == index[1])]
    r_index = rate.index
        
    if index[2] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 2] = compl_sum
        
    if index[2] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 4] = fail_sum
        
    if index[2] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 6] = term_sum
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by=["MappingMethod", "NumLayers"]))

df_rates.to_csv(r'rates_global_mapping_layers.csv', sep= ";")

,MappingMethod,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
0,densening,12,261,0.756522,2,0.005797,82,0.237681,345,1.0
1,densening,15,261,0.756522,2,0.005797,82,0.237681,345,1.0
2,densening,18,263,0.762319,4,0.011594,78,0.226087,345,1.0
3,densening,21,239,0.692754,5,0.014493,101,0.292754,345,1.0
4,densening,6,254,0.736232,3,0.008696,88,0.255072,345,1.0
5,densening,9,275,0.797101,6,0.017391,64,0.185507,345,1.0
6,direct,12,257,0.744928,4,0.011594,84,0.243478,345,1.0
7,direct,15,253,0.733333,8,0.023188,84,0.243478,345,1.0
8,direct,18,242,0.701449,7,0.02029,96,0.278261,345,1.0
9,direct,21,244,0.707246,7,0.02029,94,0.272464,345,1.0


---
#### Erfolg/Fehlerquote über je Proband je Ebenenanzahl und mapping methode

| Proband | MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED  | TERMINATED | SUM |
| ---     | ---            | ---        | ---       | ---     | ---        | --- |
| 0       | direct         | 5          | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .       | .              | .          | .         | .       | .          | .   |
| .       | .              | .          | .         | .       | .          | .   |
---

In [304]:
df_resultStat = pd.read_csv('results.csv', sep=";", names = cn_result)
df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["Proband", "MappingMethod", "NumLayers", "Completed_Abs", "Completed_Rel", "Failed_Abs", "Failed_Rel", "Terminated_Abs", "Terminated_Rel", "Sum"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers", "Result"]) 

groups = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers"])

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], elem[0][2], 0, 0, 0, 0, 0, 0, 0]

count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["Proband"] == index[0]) & (df_rates["MappingMethod"] == index[1]) & (df_rates["NumLayers"] == index[2])]
    r_index = rate.index
        
    if index[3] == " COMPLETED":
        compl_sum = int(rate["Completed_Abs"]) + value;
        df_rates.iat[r_index[0], 3] = compl_sum
        
    if index[3] == " FAILED":
        fail_sum = int(rate["Failed_Abs"]) + value;
        df_rates.iat[r_index[0], 5] = fail_sum
        
    if index[3] == " TERMINATED":
        term_sum = int(rate["Terminated_Abs"]) + value;
        df_rates.iat[r_index[0], 7] = term_sum
        
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by=["Proband", "MappingMethod", "NumLayers"]))

df_rates.to_csv(r'rates_subject_mapping_layers.csv', sep= ";")

,Proband,MappingMethod,NumLayers,Completed_Abs,Completed_Rel,Failed_Abs,Failed_Rel,Terminated_Abs,Terminated_Rel,Sum,Sum_Rel
0,0,densening,12,10,0.666667,0,0.0,5,0.333333,15,1.0
1,0,densening,15,14,0.933333,0,0.0,1,0.066667,15,1.0
2,0,densening,18,10,0.666667,0,0.0,5,0.333333,15,1.0
3,0,densening,21,10,0.666667,0,0.0,5,0.333333,15,1.0
4,0,densening,6,9,0.6,0,0.0,6,0.4,15,1.0
...,...,...,...,...,...,...,...,...,...,...,...
409,9,widening,15,4,0.266667,2,0.133333,9,0.6,15,1.0
410,9,widening,18,5,0.333333,2,0.133333,8,0.533333,15,1.0
411,9,widening,21,5,0.333333,1,0.066667,9,0.6,15,1.0
412,9,widening,6,3,0.2,5,0.333333,7,0.466667,15,1.0


---
#### Versuchsdauer über alle Probanden (nach Outcome)
| MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED | TERMINATED | AVG     |
| ---            | ---        | ---       | ---    | ---        | ---     |
| direct         | 5          | 10.145s   | 9.759s | 11.763s    | 10.345s |
| .              | .          | .         | .       | .          | .      |
| .              | .          | .         | .       | .          | .      |
---

---
#### Versuchsdauer über je Proband je Ebenenanzahl und mapping methode

| Proband | MAPPING_METHOD | NUM_LAYERS | COMPLETED | FAILED  | TERMINATED | SUM |
| ---     | ---            | ---        | ---       | ---     | ---        | --- |
| 0       | direct         | 5          | 2 / 20%   | 5 / 50% | 3 / 30%    | 10  |
| .       | .              | .          | .         | .       | .          | .   |
| .       | .              | .          | .         | .       | .          | .   |
---

In [14]:
df_resultStat = pd.read_csv('times.csv', sep=";", names = cn_times)
df_resultStat = df_resultStat.iloc[1: , :]

cn_rates = ["Proband", "MappingMethod", "NumLayers", "NumTrials", "Completed_Sum", "Completed_Avg", "Failed_Sum", "Failed_Avg", "Terminated_Sum", "Terminated_Avg", "All_Sum", "All_Avg"]
df_rates = pd.DataFrame(columns = cn_rates) 

# select all combinations of mappng method / num layers
# allConditions = df_resultStat.drop_duplicates(subset=["MappingMethod", "NumLayers"])[["MappingMethod", "NumLayers"]]

grouped = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers", "Duration"]) 

groups = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers"])

# stats = df_resultStat.groupby(["Proband", "MappingMethod", "NumLayers", "Duration"])["Duration"].mean()

# print(stats)

for elem in groups:
    df_rates.loc[len(df_rates)] = [elem[0][0], elem[0][1], elem[0][2], 0, 0, 0, 0, 0, 0, 0, 0, 0]
    
display(df_rates)

"""
count = grouped["Result"].count()

for index, value in count.items():
    rate = df_rates.loc[(df_rates["Proband"] == index[0]) & (df_rates["MappingMethod"] == index[1]) & (df_rates["NumLayers"] == index[2])]
    r_index = rate.index
        
    if index[3] == " COMPLETED":
        compl_sum = int(rate["Completed_Sum"]) + value;
        df_rates.iat[r_index[0], 4] = compl_sum
        
        
    if index[3] == " FAILED":
        fail_sum = int(rate["Failed_Sum"]) + value;
        df_rates.iat[r_index[0], 6] = fail_sum
        
    if index[3] == " TERMINATED":
        term_sum = int(rate["Terminated_Sum"]) + value;
        df_rates.iat[r_index[0], 8] = term_sum
        
    
df_rates["Sum"] = df_rates["Completed_Abs"] + df_rates["Failed_Abs"] + df_rates["Terminated_Abs"]
df_rates["Completed_Rel"] = df_rates["Completed_Abs"] / df_rates["Sum"]
df_rates["Failed_Rel"] = df_rates["Failed_Abs"] / df_rates["Sum"]
df_rates["Terminated_Rel"] = df_rates["Terminated_Abs"] / df_rates["Sum"]

df_rates["Sum_Rel"] = df_rates["Completed_Rel"] + df_rates["Failed_Rel"] + df_rates["Terminated_Rel"]
    
display(df_rates.sort_values(by=["Proband", "MappingMethod", "NumLayers"]))

df_rates.to_csv(r'times_subject_mapping_layers.csv', sep= ";")
"""

DataError: No numeric types to aggregate

* Plot: über alle Probanden für eine bestimmte Konstellation aus mapping method, Ebenenanzahl, Zielebene 
    * Y: Ebenen / Tiefe
    * X: Zeit
* Bonus: Y-Achse nach realen Tiefenwerten  skaliert (depthlayers.csv)

| PROBAND | TIMESTAMP | CURRENT_LAYER | Z     |
| ---     | ---       | ---           | ---   |
| 154     | 12345678  | 6             | -0.54 |
| .       | .         | .             | .     |
| .       | .         | .             | .     |

### Bonus
---
* depthlayers.csv: Mapping der Tiefenwerte auf die zugehörige Ebene nach Mapping-Methode Achtung: Z-Werte in Studie sind im Bereich [–1,0] zu bewerten, depthlayers.csv gibt die absolut-Werte an!
* Für erfolgreiche Trials: Jeweils zwischen HOLD und COMPLETED 
    * Maximale Schwankung (% der Ebenenbreite)
    * Durchschnittstiefe (prozentualer Abstand zur Ebenenmitte) 
    * Wie „weit“ waren die Probanden von einem Fehlschlag entfernt / wie sicher haben Sie die ebene gehalten ?
* „Wackler“ vor HOLD (wie oft wurde die Zielebene erreicht, aber nicht gehalten ?)
* Dies jeweils nach Ebenenanzahl und Mapping_Method gruppiert (über alle Probanden)
* Statistische Auswertung – Vorschläge?
* Hypothesen:
    * Sweet Spot für Anzahl der Ebenen (bspw: ab wann nimmt die Fehlerquote plötzlich stark zu ? Oder: bei welcher Ebenenanzahl wird eine Genauigkeit > 95% im Schnitt erzielt ?)
    * Vergleich der Mapping_Method: 
        * H1: widening ist genauer als direct und densening - mit zunehmender Tiefe nimmt die benötigte Kraft zu  Genauigkeit sinkt, deswegen sind die „unteren“ (aka höheren) Ebenen weiter auseinander
        * H2: densening ist genauer als direct und widening - mit zunehmender Tiefe nimmt die benötigte Kraft zu  konstante Kraftnutzung für den Ebenenwechsel = „untere“ (aka höhere) Ebenen näher beieinander
        * H3: direct ist genauer als widening und densening - Kraft spielt keine (oder nur geringfügige) Rolle, wichtig ist die äquidistante Positionierung der Tiefenebenen


In [ ]:
# your code here



In [ ]:
print(test)